# Celeb-DF v2 — Standalone Download + Batched Crop

**This notebook runs in its OWN separate Colab runtime** (parallel to the DF40 download notebook).

Workflow:
1. **Cell 1 (RUNTIME CHECK)** — confirm this is a SEPARATE runtime from the DF40 notebook before doing anything.
2. Mount Drive, set paths, restore git creds.
3. Install dlib + landmark predictor.
4. Download Celeb-DF v2 (gdown folder) to ephemeral.
5. Symlink into DeepfakeBench expected layout.
6. **Batched crop** (banks to Drive every 5 min — drop-proof).
7. rearrange.py + manifest.

**Rules learned from FF++ marathon:**
- ONE tab for THIS notebook (separate from DF40 tab is fine — different runtimes).
- NO `flush_and_unmount` ever — only `force_remount=True`.
- No gaming on the machine (bandwidth).
- Crop measured by UNIQUE VIDEO-FOLDER count, not raw frames.


## Cell 1 — RUNTIME SEPARATION CHECK (run this FIRST)

Run the SAME cell in your DF40 notebook and compare the hostnames.
- **Different hostname** → separate runtimes → parallel is SAFE, proceed.
- **Same hostname** → they share one runtime → STOP, run Celeb-DF sequentially after DF40 finishes.


In [1]:
import subprocess, os
hostname = subprocess.run("hostname", shell=True, capture_output=True, text=True).stdout.strip()
print("THIS notebook's runtime hostname:", hostname)
print()
print(">>> Run this same cell in your DF40 notebook.")
print(">>> If the hostname DIFFERS, they are separate runtimes -> safe to run both in parallel.")
print(">>> If the hostname is the SAME, STOP -> run Celeb-DF after DF40 finishes instead.")
# also show if the DF40 download is visible here (shared-disk tell)
df40_test = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research/data/df40/test_data"
# (Drive is shared across runtimes, so this being visible is EXPECTED and not a conflict signal;
#  the hostname is the real test.)
print()
print("Note: Drive files are shared across runtimes by design - that's fine.")
print("The HOSTNAME comparison is the definitive separate-runtime test.")

THIS notebook's runtime hostname: f21ea2ce3e75

>>> Run this same cell in your DF40 notebook.
>>> If the hostname DIFFERS, they are separate runtimes -> safe to run both in parallel.
>>> If the hostname is the SAME, STOP -> run Celeb-DF after DF40 finishes instead.

Note: Drive files are shared across runtimes by design - that's fine.
The HOSTNAME comparison is the definitive separate-runtime test.


## Cell 2 — Mount Drive + define paths + restore git creds

Kernel restarts wipe in-memory vars; re-run this after any reconnect.


In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, glob, subprocess

REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
PRE  = f"{REPO}/external/DeepfakeBench/preprocessing"
PARENT = "/content/drive/MyDrive/CDTS_Research"

# Celeb-DF lands here on Drive (frames)
CDF_FRAMES = f"{REPO}/data/frames/Celeb-DF-v2"
os.makedirs(CDF_FRAMES, exist_ok=True)

# restore git creds from Drive parent (per standard workflow)
for f in [".git-credentials", ".gitconfig"]:
    src = f"{PARENT}/{f}"
    if os.path.exists(src):
        subprocess.run(f'cp "{src}" /root/{f}', shell=True)
subprocess.run("git config --global credential.helper store", shell=True)

print("REPO   :", os.path.isdir(REPO))
print("PRE    :", os.path.isdir(PRE))
print("patch intact:", "return" in subprocess.run(
    f"grep -A1 'No videos found' {PRE}/preprocess.py", shell=True,
    capture_output=True, text=True).stdout and "sys.exit" not in subprocess.run(
    f"grep -A1 'No videos found' {PRE}/preprocess.py", shell=True,
    capture_output=True, text=True).stdout)

Mounted at /content/drive
REPO   : True
PRE    : True
patch intact: True


## Cell 3 — Install dlib + landmark predictor (CPU-only)

dlib HOG is CPU-bound; do NOT use a GPU runtime for this (wastes quota, GPU sits idle).
The 81-landmark predictor matches what FF++ was cropped with — keeps crops CONSISTENT across datasets.


In [3]:
import os, subprocess
# dlib-bin (prebuilt CPU wheel) — same as FF++ crop
subprocess.run("pip -q install dlib-bin==20.0.1 2>&1 | tail -2", shell=True)

# landmark predictor (same one used for FF++)
DLIB_DIR = f"{PRE}/dlib_tools"
os.makedirs(DLIB_DIR, exist_ok=True)
pred = f"{DLIB_DIR}/shape_predictor_81_face_landmarks.dat"
if not os.path.exists(pred):
    print("downloading 81-landmark predictor...")
    subprocess.run(
        f'wget -q -O "{pred}" '
        f'"https://github.com/codeniko/shape_predictor_81_face_landmarks/raw/master/shape_predictor_81_face_landmarks.dat"',
        shell=True)
print("predictor present:", os.path.exists(pred), f"({os.path.getsize(pred)/1e6:.1f}MB)" if os.path.exists(pred) else "")

import dlib
print("dlib version:", dlib.__version__)

predictor present: True (19.7MB)
dlib version: 20.0.1


## Cell 4 — Download Celeb-DF v2 to ephemeral

Folder id `1iLx76wsbi9itnkxSqz9BVBl4ZvnbIazj` (your approved link).
Downloads to ephemeral `/content/_cdf_tmp` (fast disk). May throttle — re-run to resume (`-c`).

Celeb-DF v2 contents: **Celeb-real** (590), **YouTube-real** (300), **Celeb-synthesis** (5639), **List_of_testing_videos.txt** (518-video official test split).


In [4]:
import os, glob, subprocess
CDF_RAW = "/content/_cdf_tmp"
os.makedirs(CDF_RAW, exist_ok=True)

print("=== downloading Celeb-DF v2 (re-run to resume if throttled) ===")
subprocess.run(
    f'gdown --folder "https://drive.google.com/drive/folders/1iLx76wsbi9itnkxSqz9BVBl4ZvnbIazj" '
    f'-O "{CDF_RAW}" -c 2>&1 | tail -20', shell=True)

# what came down
print("\n=== contents ===")
for sub in ["Celeb-real", "YouTube-real", "Celeb-synthesis"]:
    p = f"{CDF_RAW}/{sub}"
    n = len(glob.glob(f"{p}/*.mp4")) if os.path.isdir(p) else 0
    print(f"  {n:>5} videos  {sub}")
txt = glob.glob(f"{CDF_RAW}/**/List_of_testing_videos.txt", recursive=True)
print("  test-split file:", txt[0] if txt else "NOT FOUND")

=== downloading Celeb-DF v2 (re-run to resume if throttled) ===

=== contents ===
      0 videos  Celeb-real
      0 videos  YouTube-real
      0 videos  Celeb-synthesis
  test-split file: NOT FOUND


In [5]:
import subprocess, os, glob
CDF_RAW = "/content/_cdf_tmp"
os.makedirs(CDF_RAW, exist_ok=True)

# run gdown with FULL output visible (no tail truncation)
print("=== gdown raw output ===")
r = subprocess.run(
    f'gdown --folder "https://drive.google.com/drive/folders/1iLx76wsbi9itnkxSqz9BVBl4ZvnbIazj" -O "{CDF_RAW}" -c',
    shell=True, capture_output=True, text=True)
print("STDOUT:", r.stdout[-2000:])
print("\nSTDERR:", r.stderr[-2000:])

# what's actually in the folder now?
print("\n=== what landed ===")
for f in glob.glob(f"{CDF_RAW}/**/*", recursive=True)[:30]:
    print(" ", f)

=== gdown raw output ===
STDOUT: 

STDERR: Retrieving folder contents
Failed to retrieve folder contents


=== what landed ===


In [10]:
import os, glob, subprocess
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# find the uploaded zip (matches whatever name Drive stored it as)
hits = glob.glob("/content/drive/MyDrive/CDTS_Research/Celeb-DF*.zip") + \
       glob.glob("/content/drive/MyDrive/Celeb-DF*.zip")
print("zip in Drive:", hits)
assert hits, "Zip not found yet — drag Celeb-DF-v2.zip into My Drive/CDTS_Research and wait for upload to finish."

zip_path = hits[0]
print(f"found: {zip_path} ({os.path.getsize(zip_path)/1e9:.1f}GB)")

# unzip to ephemeral (fast disk for cropping)
CDF_RAW = "/content/_cdf_tmp"
os.makedirs(CDF_RAW, exist_ok=True)
print("unzipping... (a few minutes for 9GB)")
subprocess.run(f'unzip -q -o "{zip_path}" -d "{CDF_RAW}"', shell=True)

# show what's inside
print("\n=== extracted contents ===")
for root, dirs, files in os.walk(CDF_RAW):
    depth = root[len(CDF_RAW):].count(os.sep)
    if depth <= 1:
        mp4s = len([f for f in files if f.endswith('.mp4')])
        print(f"  {root.replace(CDF_RAW,'.')}: {mp4s} mp4" + (f", {len(dirs)} subdirs" if dirs else ""))
txt = glob.glob(f"{CDF_RAW}/**/List_of_testing_videos.txt", recursive=True)
print("test-split file:", txt[0] if txt else "NOT FOUND")
print("total mp4:", len(glob.glob(f"{CDF_RAW}/**/*.mp4", recursive=True)))

Mounted at /content/drive
zip in Drive: ['/content/drive/MyDrive/CDTS_Research/Celeb-DF-v2.zip']
found: /content/drive/MyDrive/CDTS_Research/Celeb-DF-v2.zip (10.0GB)
unzipping... (a few minutes for 9GB)

=== extracted contents ===
  .: 0 mp4, 3 subdirs
  ./Celeb-synthesis: 5639 mp4
  ./YouTube-real: 300 mp4
  ./Celeb-real: 590 mp4
test-split file: /content/_cdf_tmp/List_of_testing_videos.txt
total mp4: 6529


## Cell 5 — (OPTIONAL) Restrict to the 518-video official test split

DeepfakeBench uses Celeb-DF for **testing only** (cross-dataset eval). For trust-score work you likely
only need the **518-video official test set**, NOT all 6,529 videos. This massively cuts crop time
(~518 vs ~6,500 videos).

**Run this cell to crop ONLY the test split.** Skip it to crop everything.


In [11]:
import os, glob, shutil
CDF_RAW = "/content/_cdf_tmp"
txt = f"{CDF_RAW}/List_of_testing_videos.txt"
assert os.path.exists(txt), "test split file missing"

# parse: each line is "label path" e.g. "1 Celeb-real/id53_0003.mp4"
test_list = []
for l in open(txt):
    parts = l.split()
    if len(parts) >= 2:
        test_list.append(parts[1].strip())
print(f"official test split: {len(test_list)} videos")
# breakdown by subset
from collections import Counter
print("by subset:", dict(Counter(t.split('/')[0] for t in test_list)))

# move non-test videos aside so only the 518 test videos crop
HOLD = "/content/_cdf_hold"; os.makedirs(HOLD, exist_ok=True)
kept, held = 0, 0
for sub in ["Celeb-real", "YouTube-real", "Celeb-synthesis"]:
    src_dir = f"{CDF_RAW}/{sub}"
    if not os.path.isdir(src_dir): continue
    test_names = set(os.path.basename(t) for t in test_list if t.startswith(sub + "/"))
    os.makedirs(f"{HOLD}/{sub}", exist_ok=True)
    for v in glob.glob(f"{src_dir}/*.mp4"):
        if os.path.basename(v) in test_names:
            kept += 1
        else:
            shutil.move(v, f"{HOLD}/{sub}/{os.path.basename(v)}"); held += 1

print(f"\nkept for crop (test split): {kept} | held aside: {held}")
for sub in ["Celeb-real", "YouTube-real", "Celeb-synthesis"]:
    p = f"{CDF_RAW}/{sub}"
    n = len(glob.glob(f"{p}/*.mp4")) if os.path.isdir(p) else 0
    print(f"  {n:>4} videos to crop  {sub}")

official test split: 518 videos
by subset: {'YouTube-real': 70, 'Celeb-real': 108, 'Celeb-synthesis': 340}

kept for crop (test split): 518 | held aside: 6011
   108 videos to crop  Celeb-real
    70 videos to crop  YouTube-real
   340 videos to crop  Celeb-synthesis


## Cell 6 — Build DeepfakeBench layout (symlink + config)

DeepfakeBench expects `{root}/Celeb-DF-v2/{Celeb-real,Celeb-synthesis,YouTube-real}`.
Set preprocess config: dataset_name=`Celeb-DF-v2`, the ephemeral root, c23, 32 frames.


In [12]:
import os, glob, yaml, re
CDF_RAW = "/content/_cdf_tmp"
ROOT = "/content/_cdf_root"
os.makedirs(ROOT, exist_ok=True)
link = f"{ROOT}/Celeb-DF-v2"
if not os.path.exists(link):
    os.symlink(CDF_RAW, link)
print("symlink:", link, "->", os.path.realpath(link))
# verify the symlinked structure DeepfakeBench will see
for sub in ["Celeb-real","YouTube-real","Celeb-synthesis"]:
    n = len(glob.glob(f"{link}/{sub}/*.mp4"))
    print(f"  {n} videos visible via symlink: {sub}")

# show CURRENT config first
cfgpath = f"{PRE}/config.yaml"
c = yaml.safe_load(open(cfgpath))
print("\n=== BEFORE ===")
print("dataset_name:", c['preprocess']['dataset_name']['default'])
print("root_path   :", repr(c['preprocess']['dataset_root_path']['default']))

symlink: /content/_cdf_root/Celeb-DF-v2 -> /content/_cdf_tmp
  108 videos visible via symlink: Celeb-real
  70 videos visible via symlink: YouTube-real
  340 videos visible via symlink: Celeb-synthesis

=== BEFORE ===
dataset_name: FaceForensics++
root_path   : '/content/_raw_tmp/dfb_root'


In [13]:
import yaml
cfgpath = f"{PRE}/config.yaml"
raw = open(cfgpath).read()
raw = raw.replace("default: FaceForensics++", "default: Celeb-DF-v2")
raw = raw.replace("default: '/content/_raw_tmp/dfb_root'", "default: '/content/_cdf_root'")
open(cfgpath, "w").write(raw)
c = yaml.safe_load(open(cfgpath))
print("=== AFTER ===")
print("dataset_name:", c['preprocess']['dataset_name']['default'])
print("root_path   :", repr(c['preprocess']['dataset_root_path']['default']))

=== AFTER ===
dataset_name: FaceForensics++
root_path   : '/content/_cdf_root'


In [14]:
import subprocess
r = subprocess.run(f"grep -n 'Celeb' {PRE}/preprocess.py", shell=True, capture_output=True, text=True)
print("=== Celeb references in preprocess.py ===")
print(r.stdout if r.stdout else "NO Celeb-DF handling found")

=== Celeb references in preprocess.py ===
36:-Celeb-DF-v1/v2
37:    -Celeb-synthesis
39:    -Celeb-real
454:    ## Celeb-DF-v1
455:    elif dataset_name == 'Celeb-DF-v1':
456:        sub_dataset_names = ['Celeb-real', 'Celeb-synthesis', 'YouTube-real']
459:    ## Celeb-DF-v2
460:    elif dataset_name == 'Celeb-DF-v2':
461:        sub_dataset_names = ['Celeb-real', 'Celeb-synthesis', 'YouTube-real']



In [15]:
import yaml
cfgpath = f"{PRE}/config.yaml"
raw = open(cfgpath).read()
raw = raw.replace("default: FaceForensics++", "default: Celeb-DF-v2")
raw = raw.replace("default: '/content/_raw_tmp/dfb_root'", "default: '/content/_cdf_root'")
open(cfgpath, "w").write(raw)
c = yaml.safe_load(open(cfgpath))
print("dataset_name:", c['preprocess']['dataset_name']['default'])
print("root_path   :", repr(c['preprocess']['dataset_root_path']['default']))

dataset_name: FaceForensics++
root_path   : '/content/_cdf_root'


In [16]:
import subprocess
# find the exact dataset_name line in the preprocess section
r = subprocess.run(f"grep -n 'dataset_name' {PRE}/config.yaml", shell=True, capture_output=True, text=True)
print("=== all dataset_name lines ===")
print(r.stdout)
# show the preprocess section's dataset_name with surrounding context
r2 = subprocess.run(f"grep -n -A2 'dataset_name' {PRE}/config.yaml", shell=True, capture_output=True, text=True)
print("=== with context ===")
print(r2.stdout)

=== all dataset_name lines ===
2:  dataset_name: # the name of dataset
22:  dataset_name: # the name of dataset
41:  dataset_name: # the name of dataset

=== with context ===
2:  dataset_name: # the name of dataset
3-    choices: ['FaceForensics++','Celeb-DF-v1', 'Celeb-DF-v2', 'DFDCP', 'DFDC', 'DeeperForensics-1.0','UADFV']
4-    default: 'FaceForensics++'
--
22:  dataset_name: # the name of dataset
23-    choices: ['FaceForensics++', 'DeepFakeDetection', 'Celeb-DF-v1', 'Celeb-DF-v2','DFDCP', 'DFDC', 'DeeperForensics-1.0','UADFV','FaceShifter']
24-    default: 'FaceForensics++'
--
41:  dataset_name: # the name of dataset
42-    choices: ['FaceForensics++', 'DeepFakeDetection', 'Celeb-DF-v1', 'Celeb-DF-v2','DFDCP', 'DFDC', 'DeeperForensics-1.0','UADFV','FaceShifter']
43-    default: 'FaceForensics++'



In [17]:
import subprocess
# confirm which section each dataset_name belongs to (show section headers above)
r = subprocess.run(f"grep -n -E '^(preprocess|rearrange|to_lmdb|[a-z_]+):' {PRE}/config.yaml | head -10", shell=True, capture_output=True, text=True)
print("=== top-level sections ===")
print(r.stdout)

=== top-level sections ===
1:preprocess:
21:rearrange:
40:to_lmdb:



In [18]:
cfgpath = f"{PRE}/config.yaml"
raw = open(cfgpath).read()
idx = raw.find("default: 'FaceForensics++'")
if idx != -1:
    raw = raw[:idx] + "default: 'Celeb-DF-v2'" + raw[idx + len("default: 'FaceForensics++'"):]
open(cfgpath, "w").write(raw)

import yaml
c = yaml.safe_load(open(cfgpath))
print("preprocess dataset_name:", c['preprocess']['dataset_name']['default'])
print("preprocess root_path   :", repr(c['preprocess']['dataset_root_path']['default']))
print("rearrange dataset_name :", c['rearrange']['dataset_name']['default'], "(should stay FaceForensics++)")

preprocess dataset_name: Celeb-DF-v2
preprocess root_path   : '/content/_cdf_root'
rearrange dataset_name : FaceForensics++ (should stay FaceForensics++)


In [19]:
import subprocess, time, glob, os
CDF_RAW = "/content/_cdf_tmp"
DST = f"{REPO}/data/frames/Celeb-DF-v2"
os.makedirs(DST, exist_ok=True)

target = sum(len(glob.glob(f"{CDF_RAW}/{s}/*.mp4"))
             for s in ["Celeb-real","YouTube-real","Celeb-synthesis"] if os.path.isdir(f"{CDF_RAW}/{s}"))
print(f"cropping {target} videos (test split)\n")

os.chdir(PRE)
proc = subprocess.Popen(["python", "preprocess.py"])
while proc.poll() is None:
    time.sleep(300)
    subprocess.run(f'rsync -a --exclude="*.mp4" "{CDF_RAW}/" "{DST}/"', shell=True)
    folders = len(set(os.path.basename(f) for f in glob.glob(f"{DST}/**/frames/*", recursive=True) if os.path.isdir(f)))
    print(f"banked: {folders}/{target} video folders")
subprocess.run(f'rsync -a --exclude="*.mp4" "{CDF_RAW}/" "{DST}/"', shell=True)
os.chdir(REPO)
folders = len(set(os.path.basename(f) for f in glob.glob(f"{DST}/**/frames/*", recursive=True) if os.path.isdir(f)))
print(f"\nDONE — {folders}/{target} unique video folders on Drive")

cropping 518 videos (test split)

banked: 28/518 video folders
banked: 61/518 video folders
banked: 94/518 video folders
banked: 125/518 video folders
banked: 155/518 video folders
banked: 193/518 video folders
banked: 231/518 video folders
banked: 268/518 video folders
banked: 302/518 video folders
banked: 336/518 video folders
banked: 373/518 video folders
banked: 406/518 video folders
banked: 436/518 video folders
banked: 468/518 video folders
banked: 502/518 video folders
banked: 518/518 video folders

DONE — 518/518 unique video folders on Drive


## Cell 7 — BATCHED CROP (drop-proof, banks every 5 min)

Same pattern that landed FF++: preprocess.py in background, rsync to Drive every 5 min.
Progress measured by **unique video-folder count** (not raw frames).

If the runtime drops: re-mount (Cell 2), re-run download if ephemeral wiped (Cell 4),
then re-run this. Banked frames on Drive persist.


In [ ]:
import subprocess, time, glob, os
CDF_RAW = "/content/_cdf_tmp"
DST = f"{REPO}/data/frames/Celeb-DF-v2"
os.makedirs(DST, exist_ok=True)

# count target videos
target = sum(len(glob.glob(f"{CDF_RAW}/{s}/*.mp4"))
             for s in ["Celeb-real","YouTube-real","Celeb-synthesis"] if os.path.isdir(f"{CDF_RAW}/{s}"))
print(f"cropping ~{target} videos\n")

os.chdir(PRE)
proc = subprocess.Popen(["python", "preprocess.py"])
while proc.poll() is None:
    time.sleep(300)
    subprocess.run(f'rsync -a --exclude="*.mp4" "{CDF_RAW}/" "{DST}/"', shell=True)
    folders = len(set(os.path.basename(f)
                  for f in glob.glob(f"{DST}/**/frames/*", recursive=True) if os.path.isdir(f)))
    print(f"banked: {folders}/{target} video folders")
# final bank
subprocess.run(f'rsync -a --exclude="*.mp4" "{CDF_RAW}/" "{DST}/"', shell=True)
os.chdir(REPO)
folders = len(set(os.path.basename(f)
              for f in glob.glob(f"{DST}/**/frames/*", recursive=True) if os.path.isdir(f)))
print(f"\nDONE — {folders}/{target} unique video folders on Drive")

## Cell 8 — Verify crop completeness

Per-subset unique video-folder counts. A few no-face dropouts per subset are normal.


In [20]:
import glob, os
DST = f"{REPO}/data/frames/Celeb-DF-v2"
print("=== Celeb-DF v2 crop state (test split) ===")
total = 0
for sub in ["Celeb-real", "YouTube-real", "Celeb-synthesis"]:
    p = f"{DST}/{sub}"
    folders = len(set(os.path.basename(f) for f in glob.glob(f"{p}/**/frames/*", recursive=True) if os.path.isdir(f)))
    frames = len(glob.glob(f"{p}/**/frames/**/*.png", recursive=True))
    total += frames
    print(f"  {folders:>4} videos | {frames:>6} frames | {sub}")
print(f"  TOTAL: {total} frames across test split")

=== Celeb-DF v2 crop state (test split) ===
   108 videos |   3410 frames | Celeb-real
    70 videos |   2210 frames | YouTube-real
   340 videos |  10800 frames | Celeb-synthesis
  TOTAL: 16420 frames across test split


In [21]:
import yaml, subprocess
c = yaml.safe_load(open(f"{PRE}/config.yaml"))
print("rearrange dataset_name:", c['rearrange']['dataset_name']['default'])
print("rearrange root_path   :", repr(c['rearrange']['dataset_root_path']['default']))
r = subprocess.run(f"grep -n 'Celeb-DF-v2' {PRE}/rearrange.py", shell=True, capture_output=True, text=True)
print("\nCeleb-DF-v2 in rearrange.py:", r.stdout if r.stdout else "NOT FOUND")

rearrange dataset_name: FaceForensics++
rearrange root_path   : '/content/drive/MyDrive/CDTS_Research/deepfake-trust-research/data/frames'

Celeb-DF-v2 in rearrange.py: 297:    ## Celeb-DF-v2 dataset
299:    elif dataset_name == 'Celeb-DF-v2':



In [22]:
import subprocess
# look at the Celeb-DF-v2 rearrange branch — does it need split files or special inputs?
r = subprocess.run(f"sed -n '297,340p' {PRE}/rearrange.py", shell=True, capture_output=True, text=True)
print(r.stdout)

    ## Celeb-DF-v2 dataset
    ## Note: videos in Celeb-DF-v1/2 are not in the same format as in FaceForensics++ dataset
    elif dataset_name == 'Celeb-DF-v2':
        dataset_path = os.path.join(dataset_root_path, dataset_name)
        dataset_dict[dataset_name] = {}
        for folder in os.scandir(dataset_path):
            if not os.path.isdir(folder):
                continue
            if folder.name in ['Celeb-real', 'YouTube-real']:
                label = 'CelebDFv2_real'
            else:
                label = 'CelebDFv2_fake'
            assert label in ['CelebDFv2_real', 'CelebDFv2_fake'], 'Invalid label: {}'.format(label)
            dataset_dict[dataset_name][label] = {}
            dataset_dict[dataset_name][label]['train'] = {}
            dataset_dict[dataset_name][label]['val'] = {}
            dataset_dict[dataset_name][label]['test'] = {}
            for video_path in os.scandir(os.path.join(dataset_path, folder.name, 'frames')):
                if video_path.is

In [23]:
import shutil, os, glob
DST = f"{REPO}/data/frames/Celeb-DF-v2"
# find the test list (ephemeral) and copy to where rearrange looks (Drive frames)
src_txt = "/content/_cdf_tmp/List_of_testing_videos.txt"
dst_txt = f"{DST}/List_of_testing_videos.txt"
if os.path.exists(src_txt):
    shutil.copy(src_txt, dst_txt)
    print("copied test-list to frames folder:", os.path.exists(dst_txt))
    print("lines:", len(open(dst_txt).readlines()))
else:
    print("ephemeral test-list gone — checking Drive zip extract...")
    # fallback: it might still be in the original location
    alt = glob.glob(f"{REPO}/data/**/List_of_testing_videos.txt", recursive=True)
    print("found elsewhere:", alt)

copied test-list to frames folder: True
lines: 518


In [24]:
import yaml
cfgpath = f"{PRE}/config.yaml"
raw = open(cfgpath).read()
parts = raw.split("rearrange:")
parts[1] = parts[1].replace("default: 'FaceForensics++'", "default: 'Celeb-DF-v2'", 1)
raw = "rearrange:".join(parts)
open(cfgpath, "w").write(raw)
c = yaml.safe_load(open(cfgpath))
print("rearrange dataset_name:", c['rearrange']['dataset_name']['default'])
print("rearrange root_path   :", repr(c['rearrange']['dataset_root_path']['default']))

rearrange dataset_name: Celeb-DF-v2
rearrange root_path   : '/content/drive/MyDrive/CDTS_Research/deepfake-trust-research/data/frames'


In [25]:
import os, glob
os.chdir(PRE)
!python rearrange.py
os.chdir(REPO)
jsons = set(glob.glob(f"{PRE}/dataset_json*/Celeb*.json"))
print("\n=== Celeb-DF JSON produced ===")
for j in jsons: print(j, f"({os.path.getsize(j)} bytes)")

Celeb-DF-v2.json generated successfully.

=== Celeb-DF JSON produced ===
/content/drive/MyDrive/CDTS_Research/deepfake-trust-research/external/DeepfakeBench/preprocessing/dataset_json_v6/Celeb-DF-v2.json (6055982 bytes)


In [26]:
import sys, importlib, json, os
sys.path.insert(0, f"{REPO}/src")
import data_prep; importlib.reload(data_prep)

JSON = f"{PRE}/dataset_json_v6/Celeb-DF-v2.json"
FRAMES_ROOT = f"{REPO}/data/frames"

df = data_prep.build_manifest_from_json("Celeb-DF-v2", JSON, frames_root=FRAMES_ROOT)
print("rows (frames):", len(df))
print("\nlabel balance:\n", df['label'].value_counts())
print("\nmethod values:", df['method'].unique())
print("split values:", df['split'].unique() if 'split' in df.columns else "no split col")
print("\nunique videos:", df['video_id'].nunique(), "| unique identities:", df['identity_id'].nunique())
print("\nsample:\n", df[['video_id','label','method']].head(3).to_string())

rows (frames): 16420

label balance:
 label
1    10800
0     5620
Name: count, dtype: int64

method values: ['celeb-df-v2']
split values: ['train' 'val']

unique videos: 518 | unique identities: 124

sample:
   video_id  label       method
0    00011      0  celeb-df-v2
1    00011      0  celeb-df-v2
2    00011      0  celeb-df-v2


In [27]:
import os
manifest_path = f"{REPO}/data/celebdf_manifest.parquet"
df.to_parquet(manifest_path, index=False)
df.to_csv(f"{REPO}/data/celebdf_manifest.csv", index=False)
print("saved:", manifest_path, f"({os.path.getsize(manifest_path)/1e6:.1f}MB)")

# round-trip check
import pandas as pd
chk = pd.read_parquet(manifest_path)
print("reload:", len(chk), "rows |", chk['label'].value_counts().to_dict(), "| identities:", chk['identity_id'].nunique())

# back up the JSON to tracked location
import shutil
shutil.copy(f"{PRE}/dataset_json_v6/Celeb-DF-v2.json", f"{REPO}/data/dataset_json/Celeb-DF-v2.json")
print("JSON backed up to data/dataset_json/")

saved: /content/drive/MyDrive/CDTS_Research/deepfake-trust-research/data/celebdf_manifest.parquet (0.2MB)
reload: 16420 rows | {1: 10800, 0: 5620} | identities: 124
JSON backed up to data/dataset_json/


In [28]:
import os
os.chdir(REPO)
!cp /content/drive/MyDrive/CDTS_Research/.git-credentials /root/.git-credentials 2>/dev/null
!git add -A
!git reset src/data_prep.py.orig 2>/dev/null
print("=== staged ===")
!git status --short

^C
Unstaged changes after reset:
M	.gitignore
^C
=== staged ===
 M .gitignore
 m external/DeepfakeBench
?? data/celebdf_manifest.csv
?? data/celebdf_manifest.parquet
?? data/dataset_json/Celeb-DF-v2.json
?? data/df40/
?? notebooks/Celeb-DF_crop.ipynb


In [29]:
import os
os.chdir(REPO)
# what does git think is changed/untracked? (this shows the problem without staging)
import subprocess
r = subprocess.run("git status --short 2>&1 | head -30", shell=True, capture_output=True, text=True)
print("=== git status (first 30 lines) ===")
print(r.stdout)
r2 = subprocess.run("git status --short 2>&1 | wc -l", shell=True, capture_output=True, text=True)
print("total changed/untracked lines:", r2.stdout.strip())

=== git status (first 30 lines) ===
 M .gitignore
 m external/DeepfakeBench
?? data/celebdf_manifest.csv
?? data/celebdf_manifest.parquet
?? data/dataset_json/Celeb-DF-v2.json
?? data/df40/
?? notebooks/Celeb-DF_crop.ipynb

total changed/untracked lines: 7


In [30]:
import os
os.chdir(REPO)
with open(".gitignore", "a") as f:
    f.write("\n# DF40 data — large zips + JSONs, Drive-only\ndata/df40/\n")
print("added data/df40/ to gitignore")
import subprocess
r = subprocess.run("git check-ignore data/df40/ffpp_real.zip", shell=True, capture_output=True, text=True)
print("df40 now ignored:", bool(r.stdout.strip()))

added data/df40/ to gitignore
df40 now ignored: True


In [31]:
import os
os.chdir(REPO)
!git add data/celebdf_manifest.parquet data/celebdf_manifest.csv data/dataset_json/Celeb-DF-v2.json .gitignore notebooks/Celeb-DF_crop.ipynb
!git status --short

Refresh index: 100% (26/26), done.
M  .gitignore
A  data/celebdf_manifest.csv
A  data/celebdf_manifest.parquet
A  data/dataset_json/Celeb-DF-v2.json
 m external/DeepfakeBench
A  notebooks/Celeb-DF_crop.ipynb


In [32]:
import os
os.chdir(REPO)
!git commit -m "Add Celeb-DF-v2 test split: 518 videos (16420 frames, 124 identities) + leakage-safe manifest + JSON"
!git push origin main 2>&1 | tail -4
!git rev-parse HEAD

[main ae1d6bf] Add Celeb-DF-v2 test split: 518 videos (16420 frames, 124 identities) + leakage-safe manifest + JSON
 5 files changed, 16429 insertions(+)
 create mode 100644 data/celebdf_manifest.csv
 create mode 100644 data/celebdf_manifest.parquet
 create mode 100644 data/dataset_json/Celeb-DF-v2.json
 create mode 100644 notebooks/Celeb-DF_crop.ipynb
To https://github.com/anasbiswas1/deepfake-trust-research.git
   8eb1803..ae1d6bf  main -> main
ae1d6bf96581190feaac8bd81e1454db63846f63


## Cell 9 — RECOVERY (paste after any runtime drop)

Re-mount + redefine. Then check if ephemeral survived; if not, re-run Cell 4 (download).
**Never `flush_and_unmount` — only `force_remount`.**


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, glob, subprocess
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
PRE  = f"{REPO}/external/DeepfakeBench/preprocessing"
CDF_RAW = "/content/_cdf_tmp"
DST = f"{REPO}/data/frames/Celeb-DF-v2"

print("runtime alive (ephemeral exists):", os.path.isdir(CDF_RAW))
print("Drive Celeb-DF frames so far:",
      len(glob.glob(f"{DST}/**/frames/**/*.png", recursive=True)))
# kill any stale preprocess
subprocess.run("pkill -9 -f preprocess.py 2>/dev/null", shell=True)
print("(killed any stale preprocess processes)")
print("\nIf ephemeral is False -> re-run Cell 4 to re-download, then Cell 7 to resume crop.")